In [1]:
# Parameters
report = "results/full_report"
consolidated = "results/data/flat_consolidated.pq"
userfriendly = "results/processed_data.pq"

In [2]:
# Parameters
report = "results/full_report"
consolidated = "results/data/flat_consolidated.pq"
userfriendly = "results/processed_data.pq"


In [3]:
import pandas as pd
from IPython.display import display, Markdown

def anonymize_columns(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.replace(r'[A-Za-z0-9]', 'x', regex=True)
    return df

# Pipeline Outputs

This notebook gives an overview of the outputs generated by the pipeline. 

Outputs are placed in the `./outputs` directory.

## Human Readable Full Report

In [4]:
display(Markdown(f"""
The full report is available in the `{report}` directory as and can be viewed
be opening the [index.html]({report}/index.html) file.
"""))


The full report is available in the `results/full_report` directory as and can be viewed
be opening the [index.html](results/full_report/index.html) file.


## Consildated dataset

A flat, consilidated dataset is created by joining all tables into a single one.
Timeseries are combined into single cells, which can be exploded together again
to create a long-form dataset. 

In [5]:
consolidated_df = pd.read_parquet(consolidated)
print(consolidated_df.shape)

(14954, 638)


It uses a multilevel index for the columns and the rows. 

For columns, the first level is the dataset name and the second level is the column name.

In [6]:
display(Markdown("Datasets:"))
display(consolidated_df.columns.get_level_values(0).drop_duplicates())

display(consolidated_df.columns.to_frame().reset_index(drop=True))

# Access one dataset
followup_niere = consolidated_df["followup_niere"]

# Access one column directly
recipient_age = consolidated_df["empfaenger", "birthdate"]

Datasets:

Index(['empfaenger', 'empfaenger_dringlichkeit',
       'empfaenger_immunologie_acceptable_test',
       'empfaenger_immunologie_antibody_screening__dtt_crossmatch_nan',
       'empfaenger_immunologie_antibody_screening__dtt_crossmatch_no',
       'empfaenger_immunologie_antibody_screening__dtt_crossmatch_yes',
       'empfaenger_immunologie_hla_typing',
       'empfaenger_immunologie_unacceptable_test', 'empfaenger_virologie',
       'followup_niere', 'followup_niere_medikation', 'organ_entnahme_niere',
       'spender_postmortem', 'spender_postmortem_diagnosen',
       'spender_postmortem_labor_blutgase',
       'spender_postmortem_labor_blutgruppe',
       'spender_postmortem_labor_crossmatch', 'spender_postmortem_labor_hla',
       'spender_postmortem_labor_klinische_chemie',
       'spender_postmortem_labor_mikrobiologie',
       'spender_postmortem_labor_pathologie',
       'spender_postmortem_labor_toxikologie', 'spender_postmortem_labor_urin',
       'spender_postmortem_labor_v

,dataset,column
0,empfaenger,birthdate
1,empfaenger,bloodgroup
2,empfaenger,bloodtransfusion_after_reg
3,empfaenger,bloodtransfusion_before_reg
4,empfaenger,children
...,...,...
633,warteliste_niere,program
634,warteliste_niere,rejection_no_capacity
635,warteliste_niere,waiting_state
636,warteliste_niere,waiting_state_full


The rows are index by:

- `transplant_et_id`: The unique ET identifier for the transplantation
- `recipient_et_id_et`: The unique ET identifier for the recipient
- `donor_et_id_et`: The unique ET identifier for the donor
- `recipient_op_date`: The date of the operation for the recipient (days since the hidden reference date for the export)
- `recipient_transplant_running_id`: The running number for the recipient's transplant (always 1)

All date columns have been made relative to the `recipient_op_date` column.

In [7]:
index_rows = consolidated_df.index.to_frame().reset_index(drop=True).sample(3)
display(anonymize_columns(index_rows))

,transplant_et_id,recipient_et_id_et,donor_et_id_et,recipient_op_date,recipient_transplant_running_id
1699,xxxxxxxxxxxxxxxxxxxxxx==,xxxxxxxxxxxxxxxxxxxxxx==,xxxxxxxxxxxxxxxxxxxxxx==,2301.0,1
9047,xxxxxxxxxxx/xxxxxxxxxx==,+xxxxxxxxxxxxxxxxxxxxx==,xxxxxxxxxxxxxxx+xxxxxx==,5197.0,1
5407,x+xxxxxxxxxxxxxxxx+xxx==,xxxxxxxxxxxxxxxxxxxxxx==,xxxxxxxxxxxxxxxxxxxxxx==,1707.0,1


Here is an example of how to explode the timeseries columns within a dataset.

In [8]:
# A custom explode function, as the pandas explode() does not work with missing values
def explode_timeseries(df, cols, missing_list=[None, "", "na", "nan"]):
    df = df[cols].copy()
    lens = df.apply(lambda col: col.str.count(";;"))
    # find the length of each row (should the same or 0)
    lens_max = lens.max(axis=1, skipna=True)
    assert (lens.min(axis=1, skipna=True) == lens_max).all(), "All columns must have the same length or be empty"
    # Create the filler
    filler = (lens_max + 1).fillna(0).astype("int").apply(lambda count: ";;".join(("" for _ in range(count))))
    namask = df.isna()
    for col in df.columns:
        df.loc[namask[col], col] = filler[namask[col]]
    df = df.apply(lambda col: col.str.split(";;"))
    df = df.explode(cols).replace(missing_list, pd.NA)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors="raise")
        except ValueError:
            pass
    return df
    

Some example data from the followup dataset is shown below:

In [9]:
cols = ["date", "death_date", "death_reason", "patient_died"]
timeseries_example = followup_niere.reset_index(drop=True).loc[:, cols]
# a replacement for the actual index
new_index = [f"Patient {i+1:05}" for i in range(len(timeseries_example))]
timeseries_example["patient_id"] = new_index
timeseries_example.set_index("patient_id", inplace=True)
# drop rows with no data, only for visualization purposes
timeseries_example = timeseries_example.dropna(how="all")
timeseries_example.head()

column,date,death_date,death_reason,patient_died
patient_id,,,,
Patient 00001,467.0;;754.0;;1139.0,None,None,no;;no;;no
Patient 00002,1977.0;;365.0;;773.0;;1146.0,None,None,None;;no;;no;;no
Patient 00003,0.0;;741.0;;1106.0;;nan,None,None,None;;no;;no;;no
Patient 00004,0.0,None,None,None
Patient 00005,48.0;;342.0;;360.0,nan;;nan;;342.0,None;;None;;Infektion,None;;None;;yes


With the custom explode function, we can now explode the timeseries data while handling missing values:

In [10]:
exploded = explode_timeseries(timeseries_example, cols).sort_values(["patient_id", "date"])
exploded.head(15)

column,date,death_date,death_reason,patient_died
patient_id,,,,
Patient 00001,467.0,NaN,<NA>,no
Patient 00001,754.0,NaN,<NA>,no
Patient 00001,1139.0,NaN,<NA>,no
Patient 00002,365.0,NaN,<NA>,no
Patient 00002,773.0,NaN,<NA>,no
Patient 00002,1146.0,NaN,<NA>,no
Patient 00002,1977.0,NaN,<NA>,None
Patient 00003,0.0,NaN,<NA>,None
Patient 00003,741.0,NaN,<NA>,no


## User Friendly Dataset

Based on the configuration in the [coltypes.csv](config/coltypes.csv) file, a user-friendly dataset is created. 

Here we divided the columns into three groups: `feature`, `followup`, and `metadata`. For most time series, we made simple
decisions on what value to keep.

It uses the same indices, however the column index strucutre is different. 
If the column used to be a timeseries, it now has the suffix `__timeseries_latest` for example to indicate the performed operation.

In [11]:
userfriendly_df = pd.read_parquet(userfriendly)

display(userfriendly_df.columns.to_frame().reset_index(drop=True))

warteliste_niere_friendly = userfriendly_df.loc[:, ("warteliste_niere", slice(None), slice(None))]

warteliste_niere_friendly_features = userfriendly_df.loc[:, ("warteliste_niere", slice(None), "feature")]

,dataset,column,coltype
0,empfaenger,birthdate,feature
1,empfaenger,bloodgroup,feature
2,empfaenger,bloodtransfusion_after_reg,feature
3,empfaenger,bloodtransfusion_before_reg,feature
4,empfaenger,children,feature
...,...,...,...
633,warteliste_niere,poss_donor_sepsis__timeseries_latest,feature
634,warteliste_niere,poss_donor_substance_abuse__timeseries_latest,feature
635,warteliste_niere,waiting_state__timeseries_latest,feature
636,warteliste_niere,waiting_state_full__timeseries_latest,feature
